In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_path = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)

from utils.transformations import mold

## **DimUser**

In [0]:
##df=spark.read.format("parquet")\
##    .load("abfss://bronze@jwstorageazureproject.dfs.core.windows.net/DimUser")

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimUser/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@jwstorageazureproject.dfs.core.windows.net/DimUser")

In [0]:
display(df_user)

In [0]:
## Capitalize User name
df_user_upper=df_user.withColumn("user_name",upper(col("user_name")))

display(df_user_upper)

In [0]:
df_obj = mold()
df_user = df_obj.dropCol(df_user_upper,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])
display(df_user)

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option('path',"abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotifycata.silver.DimUser")

## **DimArtist**

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@jwstorageazureproject.dfs.core.windows.net/DimArtist")
display(df_artist)

In [0]:
df_art_obj = mold()
df_artist = df_art_obj.dropCol(df_artist,['_rescued_data'])
display(df_artist)

In [0]:
df_artist.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotifycata.silver.DimArtist")


## **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@jwstorageazureproject.dfs.core.windows.net/DimTrack")
display(df_track)

In [0]:
df_track = df_track.withColumn("durationFlag", when(col("duration_sec")<150,'low')\
                                                .otherwise(when(col("duration_sec")<300,'medium')\
                                                    .otherwise('high')))

df_track = df_track.withColumn('track_name',regexp_replace(col('track_name'),"-"," "))

df_t = mold()
df_track = df_t.dropCol(df_track,['_rescued_data'])

display(df_track)
                                                

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotifycata.silver.DimTrack")

## **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimDate/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@jwstorageazureproject.dfs.core.windows.net/DimDate")

df_date = mold().dropCol(df_date,['_rescued_data'])
display(df_date)

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@jwstorageazureproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotifycata.silver.DimDate")

## **factStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/FactStream/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@jwstorageazureproject.dfs.core.windows.net/FactStream")

df_fact = mold().dropCol(df_fact,['_rescued_data'])
display(df_fact)

In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@jwstorageazureproject.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@jwstorageazureproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotifycata.silver.FactStream")